In [1]:
import os
import random
import shutil
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import glob
from sklearn.model_selection import train_test_split
import torch
import warnings
warnings.filterwarnings('ignore')

from ultralytics import YOLO

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    memory_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"GPU Memory: {memory_gb:.1f} GB")

PyTorch version: 2.7.1+cu128
CUDA available: True
GPU: NVIDIA GeForce RTX 4070 Ti
GPU Memory: 11.6 GB


In [2]:
def load_dataset(data_dir):
    """Load dataset from all country directories"""
    dataset = []
    country_dirs = [d for d in os.listdir(data_dir) if d.startswith('country_')]
    
    for country_dir in country_dirs:
        country_path = os.path.join(data_dir, country_dir)
        images_path = os.path.join(country_path, 'images')
        labels_path = os.path.join(country_path, 'labels')

        image_files = glob.glob(os.path.join(images_path, '*.jpg'))

        for image_file in image_files:
            base_name = os.path.splitext(os.path.basename(image_file))[0]
            label_file = os.path.join(labels_path, f"{base_name}.txt")

            if os.path.exists(label_file):
                bboxes = []
                with open(label_file, 'r') as f:
                    for line in f:
                        line = line.strip()
                        if line:
                            parts = line.split()
                            if len(parts) == 5:
                                class_id = int(parts[0])
                                x_center = float(parts[1])
                                y_center = float(parts[2])
                                width = float(parts[3])
                                height = float(parts[4])
                                bboxes.append({
                                    'class_id': class_id,
                                    'x_center': x_center,
                                    'y_center': y_center,
                                    'width': width,
                                    'height': height,
                                })

                dataset.append({
                    'image_path': image_file,
                    'label_path': label_file,
                    'bboxes': bboxes,
                    'country': country_dir
                })

    return dataset

# Load dataset
data_dir = "data"
dataset = load_dataset(data_dir)

# Define class names
class_names = {
    0: 'Pothole',
    1: 'Alligator Crack',
    2: 'Transverse Crack',
    3: 'Longitudinal Crack',
}

# Basic statistics
samples_with_bboxes = [sample for sample in dataset if len(sample['bboxes']) > 0]
class_counts = {}
for sample in samples_with_bboxes:
    for bbox in sample['bboxes']:
        class_id = bbox['class_id']
        class_counts[class_id] = class_counts.get(class_id, 0) + 1

print(f"Total samples: {len(dataset)}")
print(f"Samples with annotations: {len(samples_with_bboxes)}")
print("Class distribution:")
for class_id, count in sorted(class_counts.items()):
    class_name = class_names.get(class_id, f"Class {class_id}")
    print(f"  {class_name}: {count}")

Total samples: 6039
Samples with annotations: 6039
Class distribution:
  Pothole: 3425
  Alligator Crack: 3582
  Transverse Crack: 4280
  Longitudinal Crack: 4951


In [3]:
def prepare_yolo_dataset(dataset, output_dir="yolo_dataset", train_ratio=0.8, val_ratio=0.1):
    """Prepare dataset in YOLO format with train/val/test splits"""
    # Create output directory structure
    for split in ['train', 'val', 'test']:
        os.makedirs(os.path.join(output_dir, split, 'images'), exist_ok=True)
        os.makedirs(os.path.join(output_dir, split, 'labels'), exist_ok=True)
    
    # Split dataset
    train_samples, temp_samples = train_test_split(
        samples_with_bboxes, test_size=1-train_ratio, random_state=42
    )
    val_samples, test_samples = train_test_split(
        temp_samples, test_size=val_ratio/(val_ratio + (1-train_ratio-val_ratio)), random_state=42
    )
    
    splits = {
        'train': train_samples,
        'val': val_samples,
        'test': test_samples
    }
    
    print(f"Train: {len(train_samples)}, Val: {len(val_samples)}, Test: {len(test_samples)}")
    
    # Copy files to YOLO format
    for split_name, samples in splits.items():
        for sample in samples:
            # Copy image and label
            image_name = os.path.basename(sample['image_path'])
            label_name = os.path.basename(sample['label_path'])
            
            new_image_path = os.path.join(output_dir, split_name, 'images', image_name)
            new_label_path = os.path.join(output_dir, split_name, 'labels', label_name)
            
            shutil.copy2(sample['image_path'], new_image_path)
            shutil.copy2(sample['label_path'], new_label_path)
    
    return splits, output_dir

# Prepare YOLO dataset
splits, yolo_dataset_dir = prepare_yolo_dataset(dataset)

# Create YAML config file
yaml_config = {
    'path': os.path.abspath(yolo_dataset_dir),
    'train': 'train/images',
    'val': 'val/images',
    'test': 'test/images',
    'nc': len(class_names),
    'names': list(class_names.values())
}

yaml_path = os.path.join(yolo_dataset_dir, 'data.yaml')
with open(yaml_path, 'w') as f:
    yaml.dump(yaml_config, f)

print(f"Dataset prepared: {yolo_dataset_dir}")
print(f"Config file: {yaml_path}")

Train: 4831, Val: 603, Test: 605


Dataset prepared: yolo_dataset
Config file: yolo_dataset/data.yaml


In [4]:
# Initialize YOLO11 model
model = YOLO('yolo11n.pt')

# Configure training parameters
if torch.cuda.is_available():
    device = 'cuda:0'
    batch_size = 32
    workers = 8
else:
    device = 'cpu'
    batch_size = 8
    workers = 4

# Training configuration
training_config = {
    'data': yaml_path,
    'epochs': 100,
    'batch': batch_size,
    'imgsz': 640,
    'device': device,
    'workers': workers,
    'project': 'road_damage_detection',
    'name': 'yolo11_training',
    'save_period': 10,
    'patience': 15,
    'optimizer': 'AdamW',
    'lr0': 0.01,
    'lrf': 0.01,
    'momentum': 0.937,
    'weight_decay': 0.0005,
    'warmup_epochs': 3,
    'augment': True,
    'mosaic': 1.0,
    'mixup': 0.0,
    'fliplr': 0.5,
    'hsv_h': 0.015,
    'hsv_s': 0.7,
    'hsv_v': 0.4,
    'resume': False,
    'exist_ok': True,
    'verbose': True,
    'amp': True,
}

print(f"Model: YOLO11n")
print(f"Device: {device}")
print(f"Batch size: {batch_size}")
print("Ready for training")

Model: YOLO11n
Device: cuda:0
Batch size: 32
Ready for training


In [5]:
# Start YOLO11 training
print("Starting YOLO11 training...")

try:
    results = model.train(**training_config)
    print(f"Training completed successfully!")
    print(f"Results saved at: {results.save_dir}")
    
    # Clean up GPU memory
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        
except Exception as e:
    print(f"Training failed: {e}")
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    raise e

Starting YOLO11 training...
Ultralytics 8.3.168 🚀 Python-3.13.5 torch-2.7.1+cu128 CUDA:0 (NVIDIA GeForce RTX 4070 Ti, 11885MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=True, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=yolo_dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=yolo11_training, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW, overlap_mask=True, patience=15, persp

YOLO11n summary: 181 layers, 2,590,620 parameters, 2,590,604 gradients, 6.4 GFLOPs

Transferred 448/499 items from pretrained weights
Freezing layer 'model.23.dfl.conv.weight'
AMP: running Automatic Mixed Precision (AMP) checks...
AMP: checks passed ✅
train: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2653.9±1120.2 MB/s, size: 68.0 KB)


train: Scanning /home/kwdahun/2025IMSC-Hackathon-CRACKPINK/yolo_dataset/train/labels.cache... 4831 images, 0 backgrounds, 0 corrupt: 100%|██████████| 4831/4831 [00:00<?, ?it/s]


val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1999.3±1277.5 MB/s, size: 77.5 KB)


val: Scanning /home/kwdahun/2025IMSC-Hackathon-CRACKPINK/yolo_dataset/val/labels.cache... 603 images, 0 backgrounds, 0 corrupt: 100%|██████████| 603/603 [00:00<?, ?it/s]


Plotting labels to road_damage_detection/yolo11_training/labels.jpg... 
optimizer: AdamW(lr=0.01, momentum=0.937) with parameter groups 81 weight(decay=0.0), 88 weight(decay=0.0005), 87 bias(decay=0.0)
Image sizes 640 train, 640 val
Using 8 dataloader workers
Logging results to road_damage_detection/yolo11_training
Starting training for 100 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


      1/100      4.43G      2.536      3.858      2.263         94        640:  35%|███▌      | 53/151 [00:08<00:16,  5.97it/s]


KeyboardInterrupt: 

In [10]:
# Load best model and evaluate
# Use hardcoded path based on training configuration instead of results variable
best_model_path = 'road_damage_detection/yolo11_training/weights/best.pt'
best_model = YOLO(best_model_path)

# Validate on validation set
validation_results = best_model.val(data=yaml_path, split='val')

# Test on test set
test_results = best_model.val(data=yaml_path, split='test')

# Display metrics
print("Validation Metrics:")
print(f"  mAP@0.5: {validation_results.box.map50:.4f}")
print(f"  mAP@0.5:0.95: {validation_results.box.map:.4f}")
print(f"  Precision: {validation_results.box.mp:.4f}")
print(f"  Recall: {validation_results.box.mr:.4f}")

print("\nPer-class AP@0.5:")
for i, class_name in enumerate(class_names.values()):
    if i < len(validation_results.box.ap50):
        ap50 = validation_results.box.ap50[i]
        print(f"  {class_name}: {ap50:.4f}")

Ultralytics 8.3.168 🚀 Python-3.13.5 torch-2.7.1+cu128 CUDA:0 (NVIDIA GeForce RTX 4070 Ti, 11885MiB)


YOLO11n summary (fused): 100 layers, 2,582,932 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 3358.1±885.1 MB/s, size: 81.3 KB)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 3358.1±885.1 MB/s, size: 81.3 KB)


val: Scanning /home/kwdahun/2025IMSC-Hackathon-CRACKPINK/yolo_dataset/val/labels.cache... 603 images, 0 backgrounds, 0 corrupt: 100%|██████████| 603/603 [00:00<?, ?it/s]

                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 38/38 [00:02<00:00, 13.02it/s]



                   all        603       1662      0.563      0.524      0.527      0.243
               Pothole        202        382      0.645      0.446      0.496      0.197
       Alligator Crack        286        361      0.588      0.551      0.573      0.283
      Transverse Crack        270        415      0.494      0.535       0.51      0.219
    Longitudinal Crack        307        504      0.527      0.563      0.528      0.274
Speed: 0.1ms preprocess, 1.3ms inference, 0.0ms loss, 0.8ms postprocess per image
Results saved to runs/detect/val3
               Pothole        202        382      0.645      0.446      0.496      0.197
       Alligator Crack        286        361      0.588      0.551      0.573      0.283
      Transverse Crack        270        415      0.494      0.535       0.51      0.219
    Longitudinal Crack        307        504      0.527      0.563      0.528      0.274
Speed: 0.1ms preprocess, 1.3ms inference, 0.0ms loss, 0.8ms postprocess per image
R

val: Scanning /home/kwdahun/2025IMSC-Hackathon-CRACKPINK/yolo_dataset/test/labels.cache... 605 images, 0 backgrounds, 0 corrupt: 100%|██████████| 605/605 [00:00<?, ?it/s]
val: Scanning /home/kwdahun/2025IMSC-Hackathon-CRACKPINK/yolo_dataset/test/labels.cache... 605 images, 0 backgrounds, 0 corrupt: 100%|██████████| 605/605 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 38/38 [00:02<00:00, 13.69it/s]



                   all        605       1604      0.526      0.494       0.49      0.227
               Pothole        177        344      0.532      0.334      0.389      0.159
       Alligator Crack        301        352        0.6      0.571      0.596      0.289
      Transverse Crack        269        416      0.494      0.558      0.502      0.217
    Longitudinal Crack        315        492      0.478      0.512      0.471      0.242
Speed: 0.1ms preprocess, 1.1ms inference, 0.0ms loss, 0.7ms postprocess per image
Results saved to runs/detect/val4
               Pothole        177        344      0.532      0.334      0.389      0.159
       Alligator Crack        301        352        0.6      0.571      0.596      0.289
      Transverse Crack        269        416      0.494      0.558      0.502      0.217
    Longitudinal Crack        315        492      0.478      0.512      0.471      0.242
Speed: 0.1ms preprocess, 1.1ms inference, 0.0ms loss, 0.7ms postprocess per image
R

In [11]:
def visualize_predictions(model, test_images, confidence_threshold=0.5, max_images=6):
    """Visualize model predictions on test images"""
    test_image_paths = random.sample(test_images, min(max_images, len(test_images)))
    
    n_cols = 3
    n_rows = (len(test_image_paths) + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 5*n_rows))
    axes = axes.flatten() if len(test_image_paths) > 1 else [axes]
    
    colors = ['red', 'blue', 'green', 'yellow']
    
    for i, image_path in enumerate(test_image_paths):
        results = model.predict(image_path, conf=confidence_threshold)
        
        img = Image.open(image_path)
        axes[i].imshow(img)
        
        # Draw predictions
        if len(results) > 0 and results[0].boxes is not None:
            boxes = results[0].boxes
            for box in boxes:
                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                conf = box.conf[0].cpu().numpy()
                cls = int(box.cls[0].cpu().numpy())
                
                rect = plt.Rectangle((x1, y1), x2-x1, y2-y1, 
                                   fill=False, color=colors[cls % len(colors)], 
                                   linewidth=2)
                axes[i].add_patch(rect)
                
                class_name = list(class_names.values())[cls]
                axes[i].text(x1, y1-10, f'{class_name}: {conf:.2f}',
                           bbox=dict(boxstyle="round,pad=0.3", 
                                   facecolor=colors[cls % len(colors)], 
                                   alpha=0.8),
                           fontsize=8, color='white', fontweight='bold')
        
        axes[i].set_title(f'{os.path.basename(image_path)}')
        axes[i].axis('off')
    
    # Hide unused subplots
    for i in range(len(test_image_paths), len(axes)):
        axes[i].axis('off')
    
    plt.tight_layout()
    plt.show()

# Visualize predictions on test images
test_images = [sample['image_path'] for sample in splits['test']]
visualize_predictions(best_model, test_images, confidence_threshold=0.3)


image 1/1 /home/kwdahun/2025IMSC-Hackathon-CRACKPINK/data/country_1/images/country1_007175.jpg: 640x640 1 Alligator Crack, 8.5ms
Speed: 3.0ms preprocess, 8.5ms inference, 2.1ms postprocess per image at shape (1, 3, 640, 640)
image 1/1 /home/kwdahun/2025IMSC-Hackathon-CRACKPINK/data/country_1/images/country1_007175.jpg: 640x640 1 Alligator Crack, 8.5ms
Speed: 3.0ms preprocess, 8.5ms inference, 2.1ms postprocess per image at shape (1, 3, 640, 640)


image 1/1 /home/kwdahun/2025IMSC-Hackathon-CRACKPINK/data/country_1/images/country1_006063.jpg: 640x640 2 Alligator Cracks, 7.8ms
Speed: 2.7ms preprocess, 7.8ms inference, 1.5ms postprocess per image at shape (1, 3, 640, 640)
image 1/1 /home/kwdahun/2025IMSC-Hackathon-CRACKPINK/data/country_1/images/country1_006063.jpg: 640x640 2 Alligator Cracks, 7.8ms
Speed: 2.7ms preprocess, 7.8ms inference, 1.5ms postprocess per image at shape (1, 3, 640, 640)


image 1/1 /home/kwdahun/2025IMSC-Hackathon-CRACKPINK/data/country_3/images/country3_000764.jp


image 1/1 /home/kwdahun/2025IMSC-Hackathon-CRACKPINK/data/country_1/images/country1_003412.jpg: 640x640 1 Pothole, 6.3ms
Speed: 2.2ms preprocess, 6.3ms inference, 1.2ms postprocess per image at shape (1, 3, 640, 640)
image 1/1 /home/kwdahun/2025IMSC-Hackathon-CRACKPINK/data/country_1/images/country1_003412.jpg: 640x640 1 Pothole, 6.3ms
Speed: 2.2ms preprocess, 6.3ms inference, 1.2ms postprocess per image at shape (1, 3, 640, 640)


image 1/1 /home/kwdahun/2025IMSC-Hackathon-CRACKPINK/data/country_3/images/country3_003236.jpg: 640x640 1 Pothole, 1 Longitudinal Crack, 5.9ms
Speed: 1.1ms preprocess, 5.9ms inference, 1.2ms postprocess per image at shape (1, 3, 640, 640)

image 1/1 /home/kwdahun/2025IMSC-Hackathon-CRACKPINK/data/country_3/images/country3_003236.jpg: 640x640 1 Pothole, 1 Longitudinal Crack, 5.9ms
Speed: 1.1ms preprocess, 5.9ms inference, 1.2ms postprocess per image at shape (1, 3, 640, 640)

image 1/1 /home/kwdahun/2025IMSC-Hackathon-CRACKPINK/data/country_1/images/country1

<Figure size 1500x1000 with 6 Axes>

In [12]:
# Visualize training results
# Use hardcoded path based on training configuration
results_csv = 'road_damage_detection/yolo11_training/results.csv'

if os.path.exists(results_csv):
    df = pd.read_csv(results_csv)
    df.columns = df.columns.str.strip()
    
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    
    # Plot losses
    axes[0, 0].plot(df['epoch'], df['train/box_loss'], label='Train', color='blue')
    axes[0, 0].plot(df['epoch'], df['val/box_loss'], label='Val', color='red')
    axes[0, 0].set_title('Box Loss')
    axes[0, 0].legend()
    axes[0, 0].grid(True)
    
    axes[0, 1].plot(df['epoch'], df['train/cls_loss'], label='Train', color='blue')
    axes[0, 1].plot(df['epoch'], df['val/cls_loss'], label='Val', color='red')
    axes[0, 1].set_title('Classification Loss')
    axes[0, 1].legend()
    axes[0, 1].grid(True)
    
    # Plot metrics
    axes[1, 0].plot(df['epoch'], df['metrics/mAP50(B)'], label='mAP@0.5', color='green')
    axes[1, 0].plot(df['epoch'], df['metrics/mAP50-95(B)'], label='mAP@0.5:0.95', color='orange')
    axes[1, 0].set_title('mAP')
    axes[1, 0].legend()
    axes[1, 0].grid(True)
    
    axes[1, 1].plot(df['epoch'], df['metrics/precision(B)'], label='Precision', color='purple')
    axes[1, 1].plot(df['epoch'], df['metrics/recall(B)'], label='Recall', color='brown')
    axes[1, 1].set_title('Precision & Recall')
    axes[1, 1].legend()
    axes[1, 1].grid(True)
    
    plt.tight_layout()
    plt.show()
    
    # Final metrics
    final_metrics = df.iloc[-1]
    print("Final Metrics:")
    print(f"  mAP@0.5: {final_metrics['metrics/mAP50(B)']:.4f}")
    print(f"  mAP@0.5:0.95: {final_metrics['metrics/mAP50-95(B)']:.4f}")
    print(f"  Precision: {final_metrics['metrics/precision(B)']:.4f}")
    print(f"  Recall: {final_metrics['metrics/recall(B)']:.4f}")
else:
    print("Training results not found")

<Figure size 1200x800 with 4 Axes>

Final Metrics:
  mAP@0.5: 0.5279
  mAP@0.5:0.95: 0.2414
  Precision: 0.5625
  Recall: 0.5236


In [ ]:
# Save model info (essential for evaluation)
model_info = {
    'model': 'yolo11n.pt',
    'classes': class_names,
    'best_model_path': best_model_path,
    'dataset_path': yolo_dataset_dir,
    'metrics': {
        'mAP50': float(validation_results.box.map50),
        'mAP50-95': float(validation_results.box.map),
        'precision': float(validation_results.box.mp),
        'recall': float(validation_results.box.mr)
    }
}

# Use hardcoded path based on training configuration
info_path = 'road_damage_detection/yolo11_training/model_info.yaml'
with open(info_path, 'w') as f:
    yaml.dump(model_info, f, default_flow_style=False)

print(f"Model info saved: {info_path}")

# Inference function
def predict_road_damage(image_path, confidence_threshold=0.5):
    """Predict road damage on image"""
    results = best_model.predict(image_path, conf=confidence_threshold)
    
    predictions = []
    if len(results) > 0 and results[0].boxes is not None:
        boxes = results[0].boxes
        for box in boxes:
            x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
            conf = box.conf[0].cpu().numpy()
            cls = int(box.cls[0].cpu().numpy())
            
            predictions.append({
                'class': list(class_names.values())[cls],
                'confidence': float(conf),
                'bbox': [float(x1), float(y1), float(x2), float(y2)]
            })
    
    return predictions

print("Model evaluation completed!")
print(f"Best model: {best_model_path}")
print("Use predict_road_damage(image_path) for inference")

# Optional: Export model to ONNX format (not required for evaluation)
export_onnx = False  # Set to True if you want to export to ONNX

if export_onnx:
    print("\nExporting model to ONNX format...")
    try:
        best_model.export(format='onnx')
        print("Model exported to ONNX format successfully")
    except Exception as e:
        print(f"ONNX export failed: {e}")
        print("This is optional - model evaluation works fine without ONNX export")

Exporting model...
Ultralytics 8.3.168 🚀 Python-3.13.5 torch-2.7.1+cu128 CPU (Intel Xeon Gold 6326 2.90GHz)

PyTorch: starting from 'road_damage_detection/yolo11_training/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 8, 8400) (5.2 MB)
requirements: Ultralytics requirements ['onnx>=1.12.0,<1.18.0', 'onnxslim>=0.1.59', 'onnxruntime-gpu'] not found, attempting AutoUpdate...

PyTorch: starting from 'road_damage_detection/yolo11_training/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 8, 8400) (5.2 MB)
requirements: Ultralytics requirements ['onnx>=1.12.0,<1.18.0', 'onnxslim>=0.1.59', 'onnxruntime-gpu'] not found, attempting AutoUpdate...


  error: subprocess-exited-with-error
  
  × Building wheel for onnx (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [9718 lines of output]
      fatal: not a git repository (or any of the parent directories): .git
      /tmp/pip-build-env-gco04f2m/overlay/lib/python3.13/site-packages/setuptools/config/_apply_pyprojecttoml.py:82: SetuptoolsDeprecationWarning: `project.license` as a TOML table is deprecated
      !!
      
              ********************************************************************************
              Please use a simple string containing a SPDX expression for `project.license`. You can also use `project.license-files`. (Both options available on setuptools>=77.0.0).
      
              By 2026-Feb-18, you need to update your project and remove deprecated calls
              or your builds will no longer be supported.
      
              See https://packaging.python.org/en/latest/guides/writing-pyproject-toml/#license for details.
       

WARNING ⚠️ Retry 1/2 failed: Command 'pip install --no-cache-dir "onnx>=1.12.0,<1.18.0" "onnxslim>=0.1.59" "onnxruntime-gpu" ' returned non-zero exit status 1.
WARNING ⚠️ 
ERROR ❌ ONNX: export failure 70.9s: No module named 'onnx'
Export failed: No module named 'onnx'
Model info saved: road_damage_detection/yolo11_training/model_info.yaml
Training completed!
Best model: road_damage_detection/yolo11_training/weights/best.pt
Use predict_road_damage(image_path) for inference
WARNING ⚠️ 
ERROR ❌ ONNX: export failure 70.9s: No module named 'onnx'
Export failed: No module named 'onnx'
Model info saved: road_damage_detection/yolo11_training/model_info.yaml
Training completed!
Best model: road_damage_detection/yolo11_training/weights/best.pt
Use predict_road_damage(image_path) for inference


  ERROR: Failed building wheel for onnx


ERROR: Pipe to stdout was broken
